# 5-qubit GHZ on the best qubits: VLQ vs. IQM Sirius

Kernel: **vlq_qaas100**.

**Qubit selection** uses `qubit_selector.py` (taken unchanged from another project):

* calibration comes from downloaded quality-metric-set files (`calibration_VLQ/`, `calibration_SIRIUS/`);
* `calibration_unchanged_or_confirmed` checks that the file describes the calibration set the machine is running now (asks before continuing if not);
* `rank_qubits` scores every qubit by the product of its `prx`, readout, CZ (`crf_crf`) and MOVE fidelities, ties broken by T2; the top `NUM_QB` qubits are used.

**Which qubit is logical qubit 0.** Both machines are IQM star architectures. For the GHZ circuit `H(0); CX(0,k)` the transpiler moves logical qubit 0 into the resonator (**2 × MOVE**) and every other qubit does **1 × CZ** with the resonator. With `HUB_BY_MOVE = True` the selected qubit with the best MOVE fidelity becomes logical qubit 0 (the "hub"); otherwise the best-ranked qubit does.

Then the notebook runs GHZ on the selected qubits (and on the transpiler's default qubits for reference) on both machines, compares the results with the calibration-based estimated success probability (ESP), and optionally measures the GHZ fidelity with parity oscillations.

# Settings

In [ ]:
import glob, json, os, time
from datetime import datetime, timezone
from getpass import getpass
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import hellinger_fidelity

import qubit_selector as qs

NUM_QB = 5
SHOTS = 5000

HUB_BY_MOVE = True              # logical qubit 0 = selected qubit with the best MOVE fidelity
COMPARE_DEFAULT_LAYOUT = True   # also run on the transpiler's own qubit choice (reference)
RUN_PARITY = True               # parity oscillations -> GHZ fidelity (selected qubits only)
PARITY_SHOTS = 1000

# Calibration: a folder (newest *.json in it is used) or a specific file
CALIBRATION = {"VLQ": "calibration_VLQ", "Sirius": "calibration_SIRIUS"}

# Manual override: list of NUM_QB qubit names, FIRST = logical qubit 0 (hub). None = use the selector.
VLQ_MANUAL = None      # e.g. ["QB17", "QB13", "QB22", "QB24", "QB20"]
SIRIUS_MANUAL = None

SIRIUS_USE_TIMESLOT = False
SAVE_RESULTS = True

# Connect to VLQ (LEXIS)

In [ ]:
from py4lexis.session import LexisSession
from qheappe_client.qprovider import QProvider

token = LexisSession().get_access_token()
PROJECT, RESOURCE = "OPEN-37-13", "VLQ-CZ"

vlq_provider = QProvider(lexis_project=PROJECT, token=token)
vlq_backend = vlq_provider.get_backend(lexis_resource=RESOURCE)
print("VLQ backend:", vlq_backend.name, "| qubits:", vlq_backend.num_qubits)

# Connect to IQM Sirius (IQM Resonance)

Token: `.IQM_token` file → `IQM_TOKEN` environment variable → prompt. `IQM_TOKEN` / `IQM_TOKENS_FILE` are then removed from the kernel environment, because `iqm-client` does not accept a token from both an argument and an environment variable.

In [ ]:
from iqm.qiskit_iqm import IQMProvider

TOKEN_FILE = ".IQM_token"
if os.path.exists(TOKEN_FILE):
    with open(TOKEN_FILE) as f:
        iqm_token = f.read().strip()
else:
    iqm_token = os.environ.get("IQM_TOKEN") or getpass("IQM Resonance access token: ")
for var in ("IQM_TOKEN", "IQM_TOKENS_FILE"):
    os.environ.pop(var, None)

sirius_provider = IQMProvider("https://resonance.iqm.tech/", quantum_computer="sirius", token=iqm_token)
sirius_backend = sirius_provider.get_backend()
print("Sirius backend:", sirius_backend.name, "| qubits:", sirius_backend.num_qubits)

# Load calibration files and check they match the machines

In [ ]:
BACKENDS = {"VLQ": vlq_backend, "Sirius": sirius_backend}
MANUAL = {"VLQ": VLQ_MANUAL, "Sirius": SIRIUS_MANUAL}
COLORS = {"VLQ": "#1f77b4", "Sirius": "#ff7f0e"}

def resolve_calibration_file(path):
    if os.path.isdir(path):
        files = glob.glob(os.path.join(path, "*.json"))
        if not files:
            raise FileNotFoundError(f"no *.json in {path}")
        return max(files, key=os.path.getmtime)
    return path

CAL_FILE, CAL_ID, CAL_METRICS = {}, {}, {}
for name, b in BACKENDS.items():
    CAL_FILE[name] = resolve_calibration_file(CALIBRATION[name])
    CAL_ID[name], CAL_METRICS[name] = qs.load_quality_metrics(CAL_FILE[name])
    print(f"{name:7s} {CAL_FILE[name]}\n        describes calibration set {CAL_ID[name]}")
    if not qs.calibration_unchanged_or_confirmed(SimpleNamespace(backend=b), CAL_ID[name]):
        raise RuntimeError(f"Stopped: {name} calibration changed. Download the current calibration file and re-run.")
    print("        OK - matches the machine (or confirmed)")

# Rank and select the qubits

In [ ]:
F_PRX, F_RO, F_CZ, F_MOVE = qs.FIDELITIES

def ranking_table(name):
    m = CAL_METRICS[name]
    ranked = qs.rank_qubits(m)
    rows = []
    for rank, q in enumerate(ranked, 1):
        f = [m.get(p.format(q=q)) for p in qs.FIDELITIES]
        score = np.prod([x if x is not None else 0.0 for x in f])
        rows.append({"qubit": q, "rank": rank, "F_prx": f[0], "F_readout": f[1], "F_cz": f[2], "F_move": f[3],
                     "score": score, "T2_us": (m.get(qs.T2.format(q=q)) or np.nan) * 1e6})
    return pd.DataFrame(rows).set_index("qubit")

RANK = {name: ranking_table(name) for name in BACKENDS}

# Warn when a qubit scores 0 only because its CZ uses another implementation (e.g. 'tgss' on Sirius)
for name, df in RANK.items():
    for q in df.index[df.F_cz.isna()]:
        other = [k for k in CAL_METRICS[name] if k.startswith("metrics.irb.cz.") and f".{q}__COMPR1.fidelity" in k]
        if other:
            alt = np.prod([df.loc[q, c] for c in ("F_prx", "F_readout", "F_move")]) * CAL_METRICS[name][other[0]]
            n_th = df.score.iloc[NUM_QB - 1]
            note = "WOULD be selected" if alt > n_th else "would not be selected anyway"
            print(f"NOTE {name} {q}: no crf_crf CZ field, scored 0 by the selector. "
                  f"With {other[0].split('.')[3]} CZ its score is {alt:.4f} vs. {n_th:.4f} for rank {NUM_QB} -> {note}.")

SELECTED = {}
for name, df in RANK.items():
    if MANUAL[name]:
        sel = list(MANUAL[name])
        assert len(sel) == NUM_QB and len(set(sel)) == NUM_QB, f"{name}: need {NUM_QB} distinct qubits"
        how = "manual"
    else:
        sel = list(df.index[:NUM_QB])
        how = "selector"
        if HUB_BY_MOVE:
            hub = max(sel, key=lambda q: df.loc[q, "F_move"] if pd.notna(df.loc[q, "F_move"]) else 0.0)
            sel = [hub] + [q for q in sel if q != hub]
    SELECTED[name] = sel
    print(f"{name:7s} ({how}) logical 0 (hub) = {sel[0]}, others = {sel[1:]}")

LAYOUT = {name: [BACKENDS[name].qubit_name_to_index(q) for q in SELECTED[name]] for name in BACKENDS}
print("Physical indices (logical 0..):", LAYOUT)

In [ ]:
for name, df in RANK.items():
    print(f"\n=== {name}: ranking from {CAL_FILE[name]} ===")
    sel = set(SELECTED[name])
    display(df.style
              .format({"F_prx": "{:.4f}", "F_readout": "{:.4f}", "F_cz": "{:.4f}", "F_move": "{:.4f}",
                       "score": "{:.4f}", "T2_us": "{:.1f}"}, na_rep="—")
              .apply(lambda r: ["background-color: #d4edda" if r.name in sel else "" for _ in r], axis=1))

In [ ]:
# Score per qubit (green = selected, ★ = logical qubit 0 / hub)
fig, axes = plt.subplots(len(BACKENDS), 1, figsize=(14, 3.6 * len(BACKENDS)))
for ax, name in zip(np.atleast_1d(axes), BACKENDS):
    df = RANK[name]
    colors = ["#2ca02c" if q in SELECTED[name] else "#bbbbbb" for q in df.index]
    ax.bar(range(len(df)), df.score, color=colors)
    ax.set_xticks(range(len(df)), [("★" if q == SELECTED[name][0] else "") + q for q in df.index], rotation=90)
    positive = df.score[df.score > 0]
    lo = positive.min() if len(positive) else 0
    ax.set_ylim(max(0, lo - 0.01), 1)
    ax.set_ylabel("score (product of fidelities)")
    ax.set_title(f"{name}: qubit ranking (green = selected, ★ = hub)")
plt.tight_layout()
plt.show()

# Build and transpile the GHZ circuit

The ESP of each transpiled circuit is the product of the calibrated fidelities of all its gates and measurements (CZ fidelity taken for the implementation the machine actually uses on that qubit). It ignores idling decoherence and crosstalk.

In [ ]:
from iqm.qiskit_iqm import transpile_to_IQM

def ghz_circuit(n=NUM_QB):
    c = QuantumCircuit(n)
    c.h(0)
    for qb in range(1, n):
        c.cx(0, qb)
    c.barrier()
    c.measure_all()
    return c

def transpile_for(name, circuit, initial_layout=None):
    if name == "VLQ":
        return vlq_backend.transpile(circuit, initial_layout=initial_layout)
    return transpile_to_IQM(circuit, BACKENDS[name], initial_layout=initial_layout)

def physical_qubits(name, c):
    b = BACKENDS[name]
    return [b.index_to_qubit_name(i) for i in c.layout.final_index_layout()[:NUM_QB]]

def gate_fidelity(name, gate, locus):
    m, dqa = CAL_METRICS[name], BACKENDS[name].architecture
    if gate == "prx":
        return m.get(F_PRX.format(q=locus[0]))
    if gate == "measure":
        return m.get(F_RO.format(q=locus[0]))
    if gate in ("cz", "move"):
        q, res = locus if locus[1] in dqa.computational_resonators else locus[::-1]
        impl = dqa.gates[gate].get_default_implementation((q, res))
        suffix = ":n_interleaved_gates=2" if gate == "move" else ""
        return m.get(f"metrics.irb.{gate}.{impl}.{q}__{res}.fidelity{suffix}")
    return None

def circuit_esp(name, c):
    b = BACKENDS[name]
    esp, missing, counts = 1.0, set(), {}
    for ins in c.data:
        g = {"r": "prx"}.get(ins.operation.name, ins.operation.name)
        if g in ("barrier", "delay", "id"):
            continue
        locus = tuple(b.index_to_qubit_name(c.find_bit(q).index) for q in ins.qubits)
        f = gate_fidelity(name, g, locus)
        if f is None:
            missing.add((g, locus)); continue
        esp *= f
        counts[g] = counts.get(g, 0) + 1
    return esp, counts, missing

qc = ghz_circuit()
VARIANTS = ["selected"] + (["default"] if COMPARE_DEFAULT_LAYOUT else [])
transpiled = {}
for name in BACKENDS:
    transpiled[(name, "selected")] = transpile_for(name, qc, LAYOUT[name])
    if COMPARE_DEFAULT_LAYOUT:
        transpiled[(name, "default")] = transpile_for(name, qc)

rows = []
for (name, var), c in transpiled.items():
    esp, cnt, missing = circuit_esp(name, c)
    b = BACKENDS[name]
    move_on = sorted({b.index_to_qubit_name(c.find_bit(i.qubits[0]).index) for i in c.data if i.operation.name == "move"})
    rows.append({"machine": name, "layout": var, "qubits (logical 0..)": physical_qubits(name, c), "MOVE on": move_on,
                 "depth": c.depth(), "gates": cnt, "ESP": esp, "missing calib.": sorted(missing)})
ESP_TABLE = pd.DataFrame(rows).set_index(["machine", "layout"])
display(ESP_TABLE)

In [ ]:
transpiled[("VLQ", "selected")].draw(fold=-1, idle_wires=False)

In [ ]:
transpiled[("Sirius", "selected")].draw(fold=-1, idle_wires=False)

# Run on both machines

One job per machine containing the selected-qubit circuit (and the default-layout circuit). Both jobs are submitted before waiting.

In [ ]:
def submit(name, circuits, shots):
    if name == "Sirius":
        return BACKENDS[name].run(circuits, shots=shots, use_timeslot=SIRIUS_USE_TIMESLOT)
    return BACKENDS[name].run(circuits, shots=shots)

def clean_counts(counts):
    out = {}
    for k, v in counts.items():
        k = k.replace(" ", "")
        out[k] = out.get(k, 0) + v
    return out

def collect(name, job, n_circuits):
    # VLQ (qheappe_client) needs the legacy Result to have get_counts()
    res = job.result(qiskit_legacy=True) if name == "VLQ" else job.result()
    return [clean_counts(res.get_counts(i)) for i in range(n_circuits)]

t0 = time.time()
jobs = {name: submit(name, [transpiled[(name, v)] for v in VARIANTS], SHOTS) for name in BACKENDS}
print("Submitted:", {n: str(j.job_id()) for n, j in jobs.items()})

counts = {}
for name, job in jobs.items():
    for v, c in zip(VARIANTS, collect(name, job, len(VARIANTS))):
        counts[(name, v)] = c
    print(f"{name} done after {time.time() - t0:.0f} s")

# Compare

In [ ]:
ZERO, ONE = "0" * NUM_QB, "1" * NUM_QB
ideal = {ZERO: 0.5, ONE: 0.5}

def probs(c):
    n = sum(c.values())
    return {k: v / n for k, v in c.items()}

def tvd(p, q):
    return 0.5 * sum(abs(p.get(k, 0) - q.get(k, 0)) for k in set(p) | set(q))

rows = []
for (name, v), c in counts.items():
    p, n = probs(c), sum(c.values())
    pop = p.get(ZERO, 0) + p.get(ONE, 0)
    rows.append({
        "machine": name, "layout": v,
        "qubits": ",".join(physical_qubits(name, transpiled[(name, v)])),
        "ESP (predicted)": ESP_TABLE.loc[(name, v), "ESP"],
        "P(00000)": p.get(ZERO, 0), "P(11111)": p.get(ONE, 0),
        "Population P": pop, "± P": np.sqrt(pop * (1 - pop) / n),
        "Hellinger fid.": hellinger_fidelity(c, {ZERO: n // 2, ONE: n - n // 2}),
        "TVD to ideal": tvd(p, ideal),
    })
RESULTS = pd.DataFrame(rows).set_index(["machine", "layout"])
display(RESULTS.style.format({c: "{:.4f}" for c in RESULTS.columns if c != "qubits"}))

In [ ]:
# Histogram of all runs (outcomes below 0.3 % everywhere grouped as 'other')
THRESH = 0.003
labels = [f"{n} {v}" for (n, v) in counts]
p_all = [probs(c) for c in counts.values()]
keep = sorted({k for p in p_all for k, v in p.items() if v >= THRESH})
plot_data = []
for p in p_all:
    d = {k: p.get(k, 0) for k in keep}
    d["other"] = 1 - sum(d.values())
    plot_data.append(d)
palette = {("VLQ", "selected"): "#1f77b4", ("VLQ", "default"): "#9ecae1",
           ("Sirius", "selected"): "#ff7f0e", ("Sirius", "default"): "#fdd0a2"}
fig = plot_histogram(plot_data, legend=labels, color=[palette[k] for k in counts],
                     figsize=(15, 5), bar_labels=False, title=f"{NUM_QB}-qubit GHZ, {SHOTS} shots")
fig.axes[0].set_ylabel("Probability")
display(fig)

In [ ]:
# Predicted ESP vs. measured population
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(RESULTS))
ax.bar(x - 0.2, RESULTS["ESP (predicted)"], 0.4, label="ESP (predicted from calibration)", color="#bbbbbb")
ax.bar(x + 0.2, RESULTS["Population P"], 0.4, yerr=RESULTS["± P"], label="Population P (measured)",
       color=[palette[k] for k in RESULTS.index])
ax.set_xticks(x, [f"{n}\n{v}" for n, v in RESULTS.index])
ax.set_ylim(0, 1)
ax.set_title("Calibration prediction vs. measurement")
ax.legend(fontsize=8)
plt.show()

# (Optional) Parity oscillations → GHZ fidelity on the selected qubits

Rotate every qubit by $R_z(\varphi)$ and measure in the X basis. The parity oscillates as $C\cos(N\varphi+\varphi_0)$; the fitted amplitude $C$ is the coherence and $F = (P + C)/2$ is the GHZ state fidelity ($F > 0.5$ certifies genuine $N$-qubit entanglement). $2N+2$ phases in $[0,\pi)$, linear least-squares fit of $a\cos N\varphi + b\sin N\varphi + c$, $C=\sqrt{a^2+b^2}$.

In [ ]:
def parity_circuit(phi):
    c = QuantumCircuit(NUM_QB)
    c.h(0)
    for qb in range(1, NUM_QB):
        c.cx(0, qb)
    c.barrier()
    for qb in range(NUM_QB):
        c.rz(phi, qb)
        c.h(qb)
    c.measure_all()
    return c

def parity(c):
    return sum((-1) ** k.count("1") * v for k, v in c.items()) / sum(c.values())

if RUN_PARITY:
    phis = np.linspace(0, np.pi, 2 * NUM_QB + 2, endpoint=False)
    par_circs = [parity_circuit(phi) for phi in phis]
    par_transpiled = {name: [transpile_for(name, c, LAYOUT[name]) for c in par_circs] for name in BACKENDS}
    t0 = time.time()
    par_jobs = {name: submit(name, par_transpiled[name], PARITY_SHOTS) for name in BACKENDS}
    par_counts = {}
    for name, job in par_jobs.items():
        par_counts[name] = collect(name, job, len(phis))
        print(f"{name} parity run done after {time.time() - t0:.0f} s")
else:
    print("RUN_PARITY is False, skipping.")

In [ ]:
if RUN_PARITY:
    A = np.column_stack([np.cos(NUM_QB * phis), np.sin(NUM_QB * phis), np.ones_like(phis)])
    phi_fine = np.linspace(0, np.pi, 400)
    fig, ax = plt.subplots(figsize=(8, 4))
    fidelity = {}
    for name in BACKENDS:
        y = np.array([parity(c) for c in par_counts[name]])
        (a, b, c0), *_ = np.linalg.lstsq(A, y, rcond=None)
        C = np.hypot(a, b)
        P = RESULTS.loc[(name, "selected"), "Population P"]
        fidelity[name] = {"qubits": ",".join(SELECTED[name]), "Population P": P, "Coherence C": C, "Fidelity F": (P + C) / 2}
        ax.plot(phis, y, "o", color=COLORS[name], label=f"{name} data")
        ax.plot(phi_fine, a * np.cos(NUM_QB * phi_fine) + b * np.sin(NUM_QB * phi_fine) + c0,
                "-", color=COLORS[name], label=f"{name} fit, C={C:.3f}")
    ax.plot(phi_fine, np.cos(NUM_QB * phi_fine), ":", color="gray", label="ideal")
    ax.set_xlabel(r"$\varphi$ (rad)"); ax.set_ylabel("Parity"); ax.set_ylim(-1.05, 1.05)
    ax.set_title(f"{NUM_QB}-qubit GHZ parity oscillations (selected qubits)")
    ax.legend(fontsize=8, loc="upper right")
    plt.show()
    FIDELITY = pd.DataFrame(fidelity).T
    display(FIDELITY)
    for name, f in fidelity.items():
        verdict = "genuine entanglement" if f["Fidelity F"] > 0.5 else "entanglement NOT certified"
        print(f"{name}: F = {f['Fidelity F']:.3f} -> {verdict}")

# Save results

In [ ]:
if SAVE_RESULTS:
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H-%M-%SZ")
    clean = lambda x: None if x is None or (isinstance(x, float) and np.isnan(x)) else x
    out = {
        "timestamp_utc": stamp,
        "num_qubits": NUM_QB,
        "shots": SHOTS,
        "calibration_file": CAL_FILE,
        "calibration_set_id": CAL_ID,
        "ranking": {n: [{k: clean(v) for k, v in r.items()} for r in df.reset_index().to_dict(orient="records")]
                    for n, df in RANK.items()},
        "selected_qubits": SELECTED,
        "runs": {f"{n}|{v}": {"qubits": physical_qubits(n, transpiled[(n, v)]),
                              "esp": float(ESP_TABLE.loc[(n, v), "ESP"]),
                              "counts": c}
                 for (n, v), c in counts.items()},
    }
    if RUN_PARITY:
        out["parity"] = {"phis": phis.tolist(), "shots": PARITY_SHOTS, "counts": par_counts,
                         "fidelity": {n: {k: (v if isinstance(v, str) else float(v)) for k, v in f.items()}
                                      for n, f in fidelity.items()}}
    fname = f"GHZ{NUM_QB}_best_qubits_VLQ_vs_Sirius_{stamp}.json"
    with open(fname, "w") as f:
        json.dump(out, f, indent=2, default=str)
    print("Saved", fname)